In [22]:
import pandas as pd
df = pd.read_csv('data/cleaned_online_retail.csv', parse_dates=['InvoiceDate'])

In [23]:
analysis_date = df['InvoiceDate'].max() + pd.Timedelta(days=1)

In [24]:
rfm = df.groupby('Customer ID').agg({
    'InvoiceDate': lambda x: (analysis_date - x.max()).days,
    'Invoice': 'nunique',
    'Revenue': 'sum'
}).reset_index()
rfm.columns = ['Customer ID', 'Recency', 'Frequency', 'Monetary']

In [25]:
# Définition des quintiles
r_quintiles = pd.qcut(rfm['Recency'], 5, labels=[5,4,3,2,1])
f_quintiles = pd.qcut(rfm['Frequency'].rank(method='first'), 5,
labels=[1,2,3,4,5])
m_quintiles = pd.qcut(rfm['Monetary'], 5, labels=[1,2,3,4,5])
rfm['R'] = r_quintiles.astype(int)
rfm['F'] = f_quintiles.astype(int)
rfm['M'] = m_quintiles.astype(int)
rfm['RFM_Score'] = rfm[['R','F','M']].sum(axis=1)

In [26]:
rfm

,Customer ID,Recency,Frequency,Monetary,R,F,M,RFM_Score
0,12346.0,165,11,372.86,2,5,2,9
1,12347.0,3,2,1323.32,5,2,4,11
2,12348.0,74,1,222.16,2,1,1,4
3,12349.0,43,3,2671.14,3,3,5,11
4,12351.0,11,1,300.93,5,1,2,8
...,...,...,...,...,...,...,...,...
4309,18283.0,18,6,641.77,4,5,3,12
4310,18284.0,67,1,461.68,3,2,2,7
4311,18285.0,296,1,427.00,1,2,2,5
4312,18286.0,112,2,1296.43,2,3,4,9


In [27]:
import numpy as np

conditions = [
    (rfm['R'] >= 4) & (rfm['F'] >= 4) & (rfm['M'] >= 4),  # Baleines
    (rfm['R'] >= 4) & (rfm['F'] <= 2),                     # Nouveaux clients
    (rfm['R'] <= 2) & (rfm['M'] >= 4),                     # Clients perdus
    (rfm['R'] >= 4) & (rfm['F'] >= 3),                     # Clients fidèles actifs
    (rfm['R'] <= 2) & (rfm['F'] >= 3),                     # À risque
]

labels = [
    'Baleines',
    'Nouveaux clients',
    'Clients perdus',
    'Clients fidèles actifs',
    'À risque',
]

rfm['Segment'] = np.select(conditions, labels, default='Autres')

rfm['Segment'].value_counts()

Segment
Autres                    1810
Baleines                   926
Clients fidèles actifs     467
À risque                   391
Nouveaux clients           366
Clients perdus             354
Name: count, dtype: int64

In [28]:
rfm

,Customer ID,Recency,Frequency,Monetary,R,F,M,RFM_Score,Segment
0,12346.0,165,11,372.86,2,5,2,9,À risque
1,12347.0,3,2,1323.32,5,2,4,11,Nouveaux clients
2,12348.0,74,1,222.16,2,1,1,4,Autres
3,12349.0,43,3,2671.14,3,3,5,11,Autres
4,12351.0,11,1,300.93,5,1,2,8,Nouveaux clients
...,...,...,...,...,...,...,...,...,...
4309,18283.0,18,6,641.77,4,5,3,12,Clients fidèles actifs
4310,18284.0,67,1,461.68,3,2,2,7,Autres
4311,18285.0,296,1,427.00,1,2,2,5,Autres
4312,18286.0,112,2,1296.43,2,3,4,9,Clients perdus


In [29]:
rfm.to_csv('data/rfm_segments.csv', index=False)